# Projeto 1: Análise Exploratória de Dados do Reclame Aqui - BIG Lojas

**Objetivo Geral:** Análise de Reclamações da BigLojas para Compreensão dos Problemas que Afetam o Estabelecimento.

**Objetivos Secundários:**
1. **Logística dos Produtos:** Análise desde o fornecimento até a compra e satisfação com a entrega.
2. **Qualidade do Ambiente:** Avaliação de funcionários, integridade das lojas e atendimento.

In [ ]:
# Importação das bibliotecas necessárias
import pandas as pd
import numpy as np
import re
import warnings

warnings.filterwarnings("ignore")
print("Bibliotecas importadas com sucesso!")

In [ ]:
# Carregamento do Dataset
caminho_dataset = 'RECLAMEAQUI_BIGLOJAS.csv'
df = pd.read_csv(caminho_dataset)

print(f"Dataset carregado: {df.shape[0]} linhas e {df.shape[1]} colunas.")
display(df.head(3))

## 1. Tratamento e Higienização Inicial
Padronização de textos, conversão de tipos de dados e tratamento de valores nulos para garantir a consistência estatística.

In [ ]:
# 1.1 Conversão de Datas
df['TEMPO'] = pd.to_datetime(df['TEMPO'], errors='coerce')

# 1.2 Padronização de Textos (Maiúsculas e remoção de espaços)
colunas_texto = ['TEMA', 'CATEGORIA', 'STATUS']
for col in colunas_texto:
    df[col] = df[col].str.upper().str.strip()

# 1.3 Separação de Cidade e Estado (A partir da coluna LOCAL)
df[['CIDADE', 'ESTADO']] = df['LOCAL'].str.split(' - ', expand=True, n=1)
df['CIDADE'] = df['CIDADE'].str.strip().fillna('CIDADE NÃO INFORMADA')
df['ESTADO'] = df['ESTADO'].str.strip().fillna('ESTADO NÃO INFORMADO')

# 1.4 Tratamento de Nulos
df['DESCRICAO'].fillna('SEM DESCRIÇÃO', inplace=True)

print("Limpeza inicial concluída. Status padronizados:")
print(df['STATUS'].unique())

## 2. Feature Engineering (Engenharia de Variáveis)
Criação de novas variáveis para facilitar a Análise Exploratória e alimentar os filtros do Dashboard.

In [ ]:
# Dicionário de negócio (Taxonomia)
mapeamento_temas = {
    'Problemas de Entrega e Logística': r'atraso|entrega|transportadora|frete|não recebi|faltando|veio errada',
    'Qualidade do Produto': r'estragad[oa]|vencid[oa]|podre|quebrad[oa]|defeito|mofo|mofado|bicho|inseto|larva',
    'Atendimento e Postura': r'atendimento|educação|grosseir[oa]|mal tratad[oa]|fiscal|gerente|caixa|segurança|constrangimento',
    'Precificação e Promoção': r'valor errado|diferença de preço|propaganda enganosa|oferta|desconto|etiqueta',
    'Infraestrutura da Loja': r'limpeza|estacionamento|banheiro|organização|filas|ar condicionado|carrinho',
    'Pagamento e Estorno': r'cartão|estorno|reembolso|cobrança|duplicidade|débito|hipercard',
    'Experiência Digital': r'site|aplicativo|app|ifood|online'
}

def categorizador_estrategico(linha):
    texto_completo = str(linha['CATEGORIA']).lower() + " " + str(linha['DESCRICAO']).lower() + " " + str(linha['TEMA']).lower()
    
    for tema, regex in mapeamento_temas.items():
        if re.search(regex, texto_completo):
            return tema
    return 'Outros/Geral'

# Criando a nova coluna analítica
df['CATEGORIA_ANALITICA'] = df.apply(categorizador_estrategico, axis=1)

print("Distribuição das novas categorias:")
display(df['CATEGORIA_ANALITICA'].value_counts())

## 3. Classificação Estratégica (Objetivos do Projeto)
Mapeamento das reclamações para os dois grandes objetivos do projeto: **Logística** e **Qualidade do Ambiente**.

In [ ]:
def classificar_objetivo_projeto(categoria):
    if categoria in ['Problemas de Entrega e Logística', 'Experiência Digital']:
        return 'Logística dos Produtos'
    elif categoria in ['Atendimento e Postura', 'Infraestrutura da Loja', 'Precificação e Promoção', 'Pagamento e Estorno']:
        return 'Qualidade do Ambiente'
    elif categoria == 'Qualidade do Produto':
        return 'Qualidade do Produto (Operação/Fornecedor)'
    else:
        return 'Outros/Geral'

df['OBJETIVO_PROJETO'] = df['CATEGORIA_ANALITICA'].apply(classificar_objetivo_projeto)

print("Volume de reclamações por Objetivo do Projeto:")
display(df['OBJETIVO_PROJETO'].value_counts())

In [ ]:
# Removendo colunas que não irão para o Dashboard para deixar o arquivo mais leve
colunas_remover = ['LOCAL', 'URL', 'CATEGORIA'] 
df_final = df.drop(columns=colunas_remover, errors='ignore')

# Salvando o dataset pronto
nome_arquivo = 'BIGLOJAS_DADOS_TRATADOS_FINAL.csv'
df_final.to_csv(nome_arquivo, index=False, sep=';', encoding='utf-8-sig')

print(f"Dataset salvo com sucesso como '{nome_arquivo}'. O arquivo possui {df_final.shape[0]} linhas e {df_final.shape[1]} colunas.")

## 4. Análise Exploratória de Dados (EDA)
Nesta etapa, vamos explorar visualmente os dados tratados para identificar:
1. Padrões de sazonalidade temporal.
2. Principais ofensores (Análise de Pareto das Categorias).
3. Eficácia de resolução da empresa.
4. Distribuição geográfica das ocorrências.

In [ ]:
# Importação da biblioteca Plotly Express para gráficos interativos
import plotly.express as px

print("Plotly importado com sucesso! Gráficos interativos prontos para geração.")

In [ ]:
# 4.1 Volumetria Temporal (Sazonalidade)
# Criar coluna de Ano-Mês garantindo a ordenação cronológica
df_final['ANO_MES'] = df_final['ANO'].astype(str) + '-' + df_final['MES'].astype(str).str.zfill(2)
volumetria = df_final.groupby('ANO_MES').size().reset_index(name='TOTAL_RECLAMACOES')
volumetria = volumetria.sort_values('ANO_MES')

# Criação do gráfico interativo
fig_tempo = px.line(
    volumetria, 
    x='ANO_MES', 
    y='TOTAL_RECLAMACOES', 
    markers=True,
    title='<b>Evolução Temporal das Reclamações (Sazonalidade)</b>',
    labels={'ANO_MES': 'Período (Ano-Mês)', 'TOTAL_RECLAMACOES': 'Volume de Reclamações'},
    template='plotly_white'
)

fig_tempo.update_traces(line=dict(width=3, color='#1f77b4'), marker=dict(size=8))
fig_tempo.show()

In [ ]:
# 4.2 Análise de Pareto (Principais Ofensores por Categoria Analítica)
pareto_cat = df_final['CATEGORIA_ANALITICA'].value_counts().reset_index()
pareto_cat.columns = ['CATEGORIA', 'TOTAL']

fig_pareto = px.bar(
    pareto_cat, 
    x='TOTAL', 
    y='CATEGORIA', 
    orientation='h',
    text='TOTAL',
    title='<b>Principais Ofensores: Volume por Categoria Estratégica</b>',
    labels={'TOTAL': 'Volume de Reclamações', 'CATEGORIA': 'Categoria Analítica'},
    color='TOTAL',
    color_continuous_scale='Viridis',
    template='plotly_white'
)

fig_pareto.update_layout(yaxis={'categoryorder':'total ascending'}) # Ordenar do maior para o menor no topo
fig_pareto.update_traces(textposition='outside')
fig_pareto.show()

In [ ]:
# 4.3 Eficácia de Resolução (Status das Reclamações)
status_resolucao = df_final['STATUS'].value_counts().reset_index()
status_resolucao.columns = ['STATUS', 'TOTAL']

fig_status = px.bar(
    status_resolucao, 
    x='STATUS', 
    y='TOTAL',
    text='TOTAL',
    title='<b>Eficácia de Resolução e Status Atual das Ocorrências</b>',
    labels={'STATUS': 'Status no Portal', 'TOTAL': 'Quantidade de Ocorrências'},
    color='STATUS',
    color_discrete_sequence=px.colors.qualitative.Pastel,
    template='plotly_white'
)

fig_status.update_traces(textposition='outside')
fig_status.update_layout(showlegend=False)
fig_status.show()

In [ ]:
# 4.4 Distribuição Geográfica (Top 10 Estados com mais problemas)
geo_dist = df_final['ESTADO'].value_counts().head(10).reset_index()
geo_dist.columns = ['ESTADO', 'TOTAL']

fig_geo = px.bar(
    geo_dist, 
    x='ESTADO', 
    y='TOTAL',
    text='TOTAL',
    title='<b>Gargalo Geográfico: Top 10 Estados com Maior Volume de Reclamações</b>',
    labels={'ESTADO': 'Estado (UF)', 'TOTAL': 'Quantidade de Ocorrências'},
    color='TOTAL',
    color_continuous_scale='Reds',
    template='plotly_white'
)

fig_geo.update_traces(textposition='outside')
fig_geo.show()

## 5. Diagnóstico Estratégico e Identificação de Gargalos
Nesta seção, realizamos o cruzamento de variáveis para responder a perguntas críticas de negócio baseadas nos nossos Objetivos Secundários:
1. **Gargalo Logístico:** Onde os problemas de logística estão a travar? (Cruzamento de Categoria vs. Status).
2. **Gargalo de Ambiente:** O problema nas lojas físicas é estrutural (limpeza/filas) ou humano (atendimento/postura)?

In [ ]:
# 5.1 Diagnóstico de Logística: Cruzamento de Problemas x Status de Resolução
# Filtrar apenas o objetivo de Logística
df_logistica = df_final[df_final['OBJETIVO_PROJETO'] == 'Logística dos Produtos']

# Agrupar dados
gargalo_log = df_logistica.groupby(['CATEGORIA_ANALITICA', 'STATUS']).size().reset_index(name='TOTAL')

# Criar gráfico de barras agrupadas interativo
fig_logistica = px.bar(
    gargalo_log, 
    x='CATEGORIA_ANALITICA', 
    y='TOTAL', 
    color='STATUS', 
    barmode='group',
    text='TOTAL',
    title='<b>Gargalo Logístico: Status das Reclamações de Entrega e Digital</b>',
    labels={'CATEGORIA_ANALITICA': 'Categoria Logística', 'TOTAL': 'Volume de Ocorrências'},
    color_discrete_sequence=px.colors.qualitative.Safe,
    template='plotly_white'
)

fig_logistica.update_traces(textposition='outside')
fig_logistica.update_layout(legend_title_text='Status da Reclamação')
fig_logistica.show()

In [ ]:
# 5.2 Diagnóstico de Qualidade do Ambiente: Fator Humano vs Fator Estrutural
# Filtrar apenas o objetivo de Qualidade do Ambiente
df_ambiente = df_final[df_final['OBJETIVO_PROJETO'] == 'Qualidade do Ambiente']

# Agrupar dados
gargalo_amb = df_ambiente['CATEGORIA_ANALITICA'].value_counts().reset_index()
gargalo_amb.columns = ['CATEGORIA', 'TOTAL']

# Criar gráfico de rosca (Donut Chart) interativo
fig_ambiente = px.pie(
    gargalo_amb, 
    names='CATEGORIA', 
    values='TOTAL', 
    hole=0.45, # Transforma a pizza numa rosca (Donut)
    title='<b>Raio-X da Qualidade do Ambiente nas Lojas</b>',
    color_discrete_sequence=px.colors.sequential.Plasma,
    template='plotly_white'
)

# Adicionar informações de percentagem e texto dentro do gráfico
fig_ambiente.update_traces(textposition='inside', textinfo='percent+label')
fig_ambiente.update_layout(showlegend=False) # Removemos a legenda lateral pois as labels já estão no gráfico
fig_ambiente.show()

In [ ]:
# 5.3 Heatmap de Gargalos: Categoria Analítica vs Dia da Semana
# Agrupar Categorias e Dias da Semana
heatmap_data = df_final.groupby(['CATEGORIA_ANALITICA', 'DIA_DA_SEMANA']).size().reset_index(name='TOTAL')

# Pivotar a tabela para o formato de matriz necessário para o Heatmap
matriz_heatmap = heatmap_data.pivot(index='CATEGORIA_ANALITICA', columns='DIA_DA_SEMANA', values='TOTAL').fillna(0)

# Criar o Heatmap interativo
fig_heat = px.imshow(
    matriz_heatmap,
    labels=dict(x="Dia da Semana", y="Categoria Estratégica", color="Volume"),
    x=['Segunda', 'Terça', 'Quarta', 'Quinta', 'Sexta', 'Sábado', 'Domingo'], # Mapeando os dias (1 a 7)
    y=matriz_heatmap.index,
    title='<b>Mapa de Calor (Heatmap): Concentração de Problemas por Dia da Semana</b>',
    color_continuous_scale='Reds',
    aspect="auto",
    template='plotly_white'
)

fig_heat.update_xaxes(side="top") # Coloca os dias da semana no topo para melhor leitura
fig_heat.show()

## 6. Conclusão e Plano de Ação Estratégico (Prescritivo)

Com base na Análise Exploratória (EDA) e no cruzamento de dados, diagnosticamos os principais ofensores da BIG Lojas. Atuando como uma consultoria analítica, propomos os seguintes planos de ação focados nos dois grandes objetivos do projeto:

### 🎯 Plano de Ação 1: Logística dos Produtos
**Diagnóstico (O que os dados mostram):**
Identificamos que os gargalos de entrega possuem forte correlação com a sazonalidade (ex: picos no último trimestre do ano) e que uma parcela significativa dessas ocorrências fica retida no status "Não Respondida" ou demora a ser resolvida.
**Recomendações (O que fazer):**
* **Revisão de SLAs com Transportadoras:** É imperativo renegociar contratos de *Service Level Agreement* (SLA) para prever expansão de frota terceirizada nos meses de pico (Novembro/Dezembro), mitigando o atraso na "última milha" (last mile).
* **Automação do Pós-Venda:** O alto volume de problemas logísticos "Não Respondidos" indica que o setor de atendimento está sobrecarregado. Sugerimos a implementação de *chatbots* de rastreio proativo via WhatsApp, avisando o cliente sobre o atraso antes que ele abra uma queixa no Reclame Aqui.

### 🎯 Plano de Ação 2: Qualidade do Ambiente
**Diagnóstico (O que os dados mostram):**
Ao separar problemas de infraestrutura física dos problemas de fator humano, notamos uma alta incidência de reclamações sobre **"Atendimento e Postura"** (funcionários e caixas). 
**Recomendações (O que fazer):**
* **Capacitação e Soft Skills:** O problema central das lojas não é majoritariamente estrutural, mas humano. Sugerimos o redirecionamento de parte do orçamento de marketing para a área de RH e Treinamento, com foco em empatia e resolução de conflitos na linha de frente.
* **Programa de Cliente Oculto (Mystery Shopper):** Implementar avaliações surpresa nas lojas físicas dos estados com piores índices (Gargalo Geográfico) para auditar a postura da gerência e as condições de limpeza em dias de alto fluxo (finais de semana).